In [ ]:
#Ki generierte Inhalte

import json 
from pathlib import Path 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import yfinance as yf 
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR 
from stable_baselines3 import A2C, PPO, DDPG, TD3, SAC 

In [19]:
DATA_DIR = Path("../../data") 
MODEL_DIR = Path(TRAINED_MODEL_DIR) 
VAL_DIR = Path(RESULTS_DIR) / "validation" 
OUT_DIR = Path(RESULTS_DIR) / "benchmarks" 
OUT_DIR.mkdir(parents=True, exist_ok=True) 
TRADING_DAYS = 252 
RISK_FREE_RATE = 0.0 
INDEX_TICKER = "^GDAXI" 
ALGO_CLASSES = {"a2c": A2C, "ppo": PPO, "ddpg": DDPG, "td3": TD3, "sac": SAC}

In [20]:
# Gewählten Agenten aus 03_validation laden 
selection = json.loads((VAL_DIR / "best_agent.json").read_text(encoding="utf-8")) 
AGENT = selection["name"] 
MODEL_PATH = MODEL_DIR / selection["file"]          
assert MODEL_PATH.exists(), f"Modell nicht gefunden: {MODEL_PATH.resolve()}" 
 
seed_txt = selection["seed"] if selection["seed"] is not None else "-" 
print(f"Getesteter Agent: {AGENT} ({selection['algo'].upper()}, Seed {seed_txt})") 
print(f"Gewählt per {selection['selected_by']} auf Validierung {selection['val_period'][0]} bis " 
      f"{selection['val_period'][1]} | Val-Sharpe {selection['val_metrics']['Sharpe']:.2f}")

Getesteter Agent: sac_seed_0 (SAC, Seed -)
Gewählt per Sharpe auf Validierung 2020-09-01 bis 2021-08-31 | Val-Sharpe 1.52


In [21]:
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv 
 
class TRStockTradingEnv(StockTradingEnv): 
    """StockTradingEnv mit Trade-Republic-Fixgebuehr von 1 EUR je Ausfuehrung. 
 
    Die geerbten buy_cost_pct / sell_cost_pct bleiben als Spread aktiv. 
    Quelle der Kondition: Preis- und Leistungsverzeichnis Trade Republic. 
    """ 
 
    def __init__(self, *args, fixed_fee=1.0, **kwargs): 
        super().__init__(*args, **kwargs) 
        self.fixed_fee = fixed_fee 
        self.fee_total = 0.0 
 
    def _charge_fee(self, shares): 
        if shares > 0: 
            self.state[0] -= self.fixed_fee   # Cash belasten 
            self.cost += self.fixed_fee       # in FinRLs Kostenzaehler 
            self.fee_total += self.fixed_fee  # eigener Zaehler fuer die Auswertung 
 
    def _buy_stock(self, index, action): 
        shares = super()._buy_stock(index, action) 
        self._charge_fee(shares) 
        return shares 
 
    def _sell_stock(self, index, action): 
        shares = super()._sell_stock(index, action) 
        self._charge_fee(shares) 
        return shares 
 
    def reset(self, *, seed=None, options=None): 
        out = super().reset(seed=seed, options=options) 
        self.fee_total = 0.0 
        return out

In [22]:
def load_like_train(file): 
    df = pd.read_csv(DATA_DIR / file) 
    date = df.columns[0] 
    df.index = pd.factorize(df[date])[0] 
    df.index.names = [''] 
    return df 
 
train = load_like_train("train.csv") 
test = load_like_train("trade.csv") 
assert list(train.tic.unique()) == list(test.tic.unique()) 
 
stock_dimensions = len(test.tic.unique()) 
state_space = 1 + 2 * stock_dimensions + len(INDICATORS) * stock_dimensions 
print(f"Test: {test.date.min()} bis {test.date.max()} | {test.date.nunique()} Handelstage | " 
      f"{stock_dimensions} Aktien | State Space {state_space}")

Test: 2021-09-01 bis 2026-09-03 | 1241 Handelstage | 31 Aktien | State Space 311


In [23]:
#Env-Parameter – 1:1 wie in 02_train / 03_validation 
buy_cost_list = sell_cost_list = [0.0005] * stock_dimensions 
num_stock_shares = [0] * stock_dimensions 
 
env_tweeks = { 
    "hmax": 50, 
    "initial_amount": 100000, 
    "num_stock_shares": num_stock_shares, 
    "buy_cost_pct": buy_cost_list, 
    "sell_cost_pct": sell_cost_list, 
    "state_space": state_space, 
    "stock_dim": stock_dimensions, 
    "tech_indicator_list": INDICATORS, 
    "action_space": stock_dimensions, 
    "reward_scaling": 1e-4 
} 
FIXED_FEE = 1.0 
START = env_tweeks["initial_amount"] 
SPREAD = buy_cost_list[0]

In [24]:
 #Kennzahlen – identisch zu 03_validation 
def kennzahlen(values: pd.Series) -> dict: 
    r = values.pct_change().dropna() 
    years = len(r) / TRADING_DAYS 
    excess = r - RISK_FREE_RATE / TRADING_DAYS 
    cagr = (values.iloc[-1] / values.iloc[0]) ** (1 / years) - 1 
    vola = r.std() * np.sqrt(TRADING_DAYS) 
    sharpe_d = excess.mean() / r.std() if r.std() > 0 else np.nan 
    downside = np.sqrt((np.minimum(excess, 0) ** 2).mean()) 
    mdd = (values / values.cummax() - 1).min() 
    return { 
        "Endwert": values.iloc[-1], 
        "Rendite": values.iloc[-1] / values.iloc[0] - 1, 
        "CAGR": cagr, 
        "Vola": vola, 
        "Sharpe": sharpe_d * np.sqrt(TRADING_DAYS), 
        "Sharpe_SE": np.sqrt((1 + 0.5 * sharpe_d ** 2) / len(r)) * np.sqrt(TRADING_DAYS), 
        "Sortino": excess.mean() / downside * np.sqrt(TRADING_DAYS) if downside > 0 else np.nan, 
        "MaxDD": mdd, 
        "Calmar": cagr / abs(mdd) if mdd < 0 else np.nan, 
    } 

In [25]:
#  Benchmarks – identisch zu 03_validation (gleiche Kosten + Startkapital) 
px = test.pivot(index="date", columns="tic", values="close").sort_index() 
px.index = pd.to_datetime(px.index) 
px = px.replace(0, np.nan).ffill().dropna(axis=1, how="any") 
 
def equity(px, rebal_days, start=START, fee=FIXED_FEE, spread=SPREAD): 
    """Depotwert je Handelstag. Rebalancing auf Gleichgewichtung an rebal_days.""" 
    n, rebal_days = px.shape[1], set(rebal_days) 
    shares, cash, out = pd.Series(0.0, index=px.columns), start, [] 
    for d in px.index: 
        p = px.loc[d] 
        if d in rebal_days: 
            value = cash + (shares * p).sum() 
            trade_eur = ((value / n) / p - shares).abs() * p 
            orders = int((trade_eur > 1.0).sum()) 
            value -= orders * fee + trade_eur.sum() * spread 
            shares, cash = (value / n) / p, 0.0 
        out.append(cash + (shares * p).sum()) 
    return pd.Series(out, index=px.index) 
 
def load_index(ticker, dates, start=START): 
    """Index von Yahoo laden und auf Startkapital normieren. None, wenn der Download scheitert.""" 
    try: 
        raw = yf.download(ticker, start=dates[0].strftime("%Y-%m-%d"), 
                          end=(pd.Timestamp(dates[-1]) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"), 
                          auto_adjust=True, progress=False)["Close"] 
        s = raw.squeeze() 
        s.index = pd.to_datetime(s.index) 
        if s.index.tz is not None: 
            s.index = s.index.tz_localize(None) 
        s = s.reindex(dates).ffill().bfill() 
        if s.isna().all(): 
            raise ValueError("keine Kurse") 
        return s / s.iloc[0] * start 
    except Exception as exc: 
        print(f"{ticker} konnte nicht geladen werden ({exc}) -> ohne Index weiter") 
        return None 
 
monats_erste = px.groupby(px.index.to_period("M")).head(1).index 
 
benchmarks = { 
    "1/N halten":    equity(px, rebal_days=[px.index[0]]), 
    "1/N monatlich": equity(px, rebal_days=monats_erste), 
    "DAX":           load_index(INDEX_TICKER, px.index), 
} 
benchmarks = pd.DataFrame({k: v for k, v in benchmarks.items() if v is not None}) 
display(benchmarks.tail(3))

,1/N halten,1/N monatlich,DAX
date,,,
2026-09-01,216839.352372,178787.771612,164115.478868
2026-09-02,216787.639527,177853.064053,163289.032332
2026-09-03,217376.386165,178507.499867,164325.352027


In [26]:
#Agent EINMAL deterministisch über den Testzeitraum laufen lassen
model = ALGO_CLASSES[selection["algo"]].load(str(MODEL_PATH), device="cpu") 
assert model.observation_space.shape[0] == state_space, "State-Größe passt nicht zum Modell" 
env = TRStockTradingEnv(df=test, fixed_fee=FIXED_FEE, print_verbosity=10**9, **env_tweeks) 
obs, _ = env.reset() 
done = False 
while not done: 
    action, _ = model.predict(np.asarray(obs, dtype=np.float32), deterministic=True) 
    obs, _, terminated, truncated, _ = env.step(action) 
    done = terminated or truncated 
account = env.save_asset_memory() 
agent_values = pd.Series(account["account_value"].to_numpy(), 
index=pd.to_datetime(account["date"]), name=AGENT) 
agent_stats = {"Orders": int(round(env.fee_total / FIXED_FEE)), "Gebuehren": 
env.fee_total, 
    "Kosten": env.cost, "Cash_Ende": float(env.state[0])} 
values = pd.concat([benchmarks, agent_values], axis=1).sort_index().ffill() 
print(f"{AGENT}: Endwert {agent_values.iloc[-1]:,.0f} | Orders {agent_stats['Orders']} | " 
    f"Gebühren {agent_stats['Gebuehren']:,.0f} | Kosten gesamt {agent_stats['Kosten']:,.0f}")

sac_seed_0: Endwert 132,811 | Orders 316 | Gebühren 316 | Kosten gesamt 458


In [27]:
metrics = pd.DataFrame({col: kennzahlen(values[col].dropna()) for col in values}).T 
metrics = metrics.join(pd.DataFrame({AGENT: agent_stats}).T) 
metrics["Typ"] = np.where(metrics.index == AGENT, "Agent", "Benchmark") 

pd.options.display.float_format = "{:,.3f}".format 
display(metrics.sort_values("Sharpe", ascending=False)) 
best_bench_name = metrics.loc[benchmarks.columns, "Sharpe"].astype(float).idxmax() 
diff = metrics.loc[AGENT, "Sharpe"] - metrics.loc[best_bench_name, "Sharpe"] 
se = metrics.loc[AGENT, "Sharpe_SE"] 
print(f"Sharpe {AGENT}: {metrics.loc[AGENT, 'Sharpe']:.2f} ± {se:.2f} | bester Benchmark "
f"{best_bench_name}: {metrics.loc[best_bench_name, 'Sharpe']:.2f} | Differenz {diff:+.2f}") 
if diff <= 0: 
    print("-> Agent schlägt den besten Benchmark im Testzeitraum NICHT.") 
elif diff < se: 
    print("-> Agent vorn, aber Vorsprung kleiner als 1 Standardfehler: statistisch nicht belastbar.") 
else: 
    print("-> Agent schlägt den besten Benchmark um mehr als 1 Standardfehler.")

,Endwert,Rendite,CAGR,Vola,Sharpe,Sharpe_SE,Sortino,MaxDD,Calmar,Orders,Gebuehren,Kosten,Cash_Ende,Typ
1/N halten,"217,376.386",1.176,0.171,0.178,0.975,0.451,1.434,-0.183,0.935,NaN,NaN,NaN,NaN,Benchmark
1/N monatlich,"178,507.500",0.787,0.125,0.157,0.829,0.451,1.199,-0.199,0.628,NaN,NaN,NaN,NaN,Benchmark
DAX,"164,325.352",0.643,0.106,0.173,0.670,0.451,0.964,-0.264,0.402,NaN,NaN,NaN,NaN,Benchmark
sac_seed_0,"132,811.166",0.328,0.059,0.187,0.402,0.451,0.582,-0.254,0.234,316.000,316.000,458.356,1.413,Agent


Sharpe sac_seed_0: 0.40 ± 0.45 | bester Benchmark 1/N halten: 0.98 | Differenz -0.57
-> Agent schlägt den besten Benchmark im Testzeitraum NICHT.


In [28]:
#Validierung vs. Test – Overfitting-Check
cols = ["Rendite", "CAGR", "Vola", "Sharpe", "Sortino", "MaxDD", "Calmar"] 
val_vs_test = pd.DataFrame({ 
"Validierung": {k: selection["val_metrics"][k] for k in cols}, 
"Test": {k: float(metrics.loc[AGENT, k]) for k in cols}, 
}) 
val_vs_test["Differenz"] = val_vs_test["Test"] - val_vs_test["Validierung"] 
display(val_vs_test) 
# Deutlich schlechtere Test- als Val-Werte = Agent hat sich (über Auswahl/Training) an den Validierungszeitraum angepasst oder das Marktregime hat gewechselt. 


,Validierung,Test,Differenz
Rendite,0.263,0.328,0.065
CAGR,0.269,0.059,-0.209
Vola,0.166,0.187,0.020
Sharpe,1.516,0.402,-1.114
Sortino,2.178,0.582,-1.596
MaxDD,-0.114,-0.254,-0.139
Calmar,2.349,0.234,-2.115


In [29]:
#Plots: Equity, Drawdown, rollierende Sharpe
COLORS = {"1/N halten": "#0072B2", "1/N monatlich": "#E69F00", "DAX": "#009E73", 
AGENT: "#D55E00"} 
 
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True) 
for col in values.columns: 
    is_agent = col == AGENT 
    style = dict(color=COLORS.get(col, "#666666"), linestyle="-" if is_agent else "--", 
                 linewidth=3 if is_agent else 2, label=col) 
    axes[0].plot(values[col] / values[col].iloc[0] * 100, **style) 
    axes[1].plot(values[col] / values[col].cummax() - 1, **style) 
axes[0].set(title=f"Test: Portfolioentwicklung (Start = 100) – {AGENT}", ylabel="Indexierter Wert") 
axes[1].set(title="Test: Drawdown", ylabel="Drawdown", xlabel="Datum") 
for ax in axes: 
    ax.grid(alpha=0.3) 
    ax.legend(loc="upper left") 
fig.tight_layout() 
fig.savefig(OUT_DIR / "equity_drawdown_test.png", dpi=150, bbox_inches="tight") 
plt.show() 
 
# Rollierende 1-Jahres-Sharpe: ist der Agent dauerhaft besser oder nur in einer Phase? 
rets = values.pct_change() 
rolling_sharpe = rets.rolling(TRADING_DAYS).mean() / rets.rolling(TRADING_DAYS).std() * np.sqrt(TRADING_DAYS) 
fig, ax = plt.subplots(figsize=(15, 5)) 
for col in values.columns: 
    ax.plot(rolling_sharpe[col], color=COLORS.get(col, "#666666"), 
            linewidth=2.5 if col == AGENT else 1.5, linestyle="-" if col == AGENT else "--", label=col) 
ax.axhline(0, color="black", lw=0.8) 
ax.set(title="Test: rollierende Sharpe Ratio (252 Tage)", ylabel="Sharpe", xlabel="Datum") 
ax.grid(alpha=0.3) 
ax.legend(loc="upper left") 
fig.tight_layout() 
fig.savefig(OUT_DIR / "rolling_sharpe_test.png", dpi=150, bbox_inches="tight") 
plt.show()

C:\Users\fabia\AppData\Local\Temp\ipykernel_20792\340879613.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\fabia\AppData\Local\Temp\ipykernel_20792\340879613.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
#Plots: Jahresrenditen + Kennzahlen
yearly = values.resample("YE").last().pct_change() 
yearly.iloc[0] = values.resample("YE").last().iloc[0] / values.iloc[0] - 1     # erstes (Teil)Jahr 
yearly.index = yearly.index.year 
ax = yearly.plot(kind="bar", figsize=(12, 5), color=[COLORS.get(c, "#666666") for c in 
yearly.columns]) 
ax.axhline(0, color="black", lw=0.8) 
ax.set(title="Test: Rendite je Kalenderjahr", ylabel="Rendite", xlabel="Jahr") 
ax.grid(axis="y", alpha=0.3) 
plt.tight_layout() 
plt.savefig(OUT_DIR / "yearly_returns_test.png", dpi=150, bbox_inches="tight") 
plt.show() 
order = metrics.sort_values("Sharpe").index 
fig, axes = plt.subplots(1, 3, figsize=(16, 0.5 * len(order) + 2), sharey=True) 
for ax, col in zip(axes, ["Sharpe", "CAGR", "MaxDD"]): 
    xerr = metrics.loc[order, "Sharpe_SE"].astype(float) if col == "Sharpe" else None 
    ax.barh(order, metrics.loc[order, col].astype(float), xerr=xerr, capsize=3, color=[COLORS.get(n, "#666666") for n in order]) 
    ax.set_title(col + (" ± 1 SE" if col == "Sharpe" else "")) 
    ax.grid(axis="x", alpha=0.3) 
fig.tight_layout() 
fig.savefig(OUT_DIR / "metrics_test.png", dpi=150, bbox_inches="tight") 
plt.show()

C:\Users\fabia\AppData\Local\Temp\ipykernel_20792\2227063071.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\fabia\AppData\Local\Temp\ipykernel_20792\2227063071.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
#Portfolio des Agenten am Ende des Testzeitraums
cash = float(env.state[0]) 
prices = np.asarray(env.state[1:1 + stock_dimensions], dtype=float) 
shares = np.asarray(env.state[1 + stock_dimensions:1 + 2 * stock_dimensions], 
dtype=float) 
total = cash + (prices * shares).sum() 
 
composition = pd.DataFrame({"Aktie": test.tic.drop_duplicates().tolist(), "Stueck": 
shares, 
                            "Kurs": prices, "Wert": prices * shares}) 
composition["Anteil_Prozent"] = composition["Wert"] / total * 100 
composition = composition[composition["Stueck"] > 0].sort_values("Wert", 
ascending=False).reset_index(drop=True) 
 
print(f"Cash: {cash:,.2f} ({cash / total:.1%}) | Gesamtwert: {total:,.2f} | " 
      f"Positionen: {len(composition)} von {stock_dimensions}") 
display(composition)

Cash: 1.41 (0.0%) | Gesamtwert: 132,811.17 | Positionen: 10 von 31


,Aktie,Stueck,Kurs,Wert,Anteil_Prozent
0,RHM.DE,31.000,"1,070.200","33,176.198",24.980
1,VOW3.DE,370.000,76.360,"28,253.200",21.273
2,G1A.DE,279.000,64.100,"17,883.900",13.466
3,RWE.DE,276.000,57.980,"16,002.480",12.049
4,IFX.DE,235.000,55.530,"13,049.550",9.826
5,QIA.DE,189.000,37.470,"7,081.830",5.332
6,DHL.DE,127.000,55.200,"7,010.400",5.278
7,FRE.DE,151.000,44.225,"6,677.975",5.028
8,FME.DE,77.000,38.600,"2,972.200",2.238
9,ADS.DE,7.000,148.700,"1,040.900",0.784


In [32]:
#Export  
values.to_csv(OUT_DIR / "portfolio_values_test.csv") 
metrics.to_csv(OUT_DIR / "metrics_test.csv") 
val_vs_test.to_csv(OUT_DIR / "val_vs_test.csv") 
composition.to_csv(OUT_DIR / "composition_test.csv", index=False) 
(OUT_DIR / "test_result.json").write_text(json.dumps({ 
    "agent": selection, 
    "test_period": [str(test.date.min()), str(test.date.max())], 
    "test_metrics": {k: float(metrics.loc[AGENT, k]) for k in cols + ["Endwert", "Sharpe_SE"]}, 
"test_costs": agent_stats, 
"best_benchmark": {"name": best_bench_name, "Sharpe": 
float(metrics.loc[best_bench_name, "Sharpe"])}, 
}, indent=2, ensure_ascii=False), encoding="utf-8") 
print(f"Gespeichert in {OUT_DIR.resolve()}")

Gespeichert in C:\Users\fabia\Desktop\FinRl\pro-finrl\src\notebooks\Alt\results\benchmarks
